# Project 12: Climate Risk Scenarios

Climate transition and physical risk quantification aligned with NGFS pathways and TCFD reporting.

This notebook walks through:
1. Loading NGFS scenarios and visualizing carbon price and temperature paths
2. Sector carbon intensity overview
3. Transition risk: sector repricing under each scenario
4. Physical risk: damage function curve and sector exposure
5. Climate VaR: combined risk across scenarios
6. Sobol sensitivity: which climate factor dominates?
7. TCFD metrics and report generation

In [ ]:
import sys
from pathlib import Path

# Add project src to path
sys.path.insert(0, str(Path(".").resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load NGFS Scenarios

We load six NGFS Phase V pathways. In production, these would be downloaded from the IIASA database. Here we use synthetic data calibrated to approximate NGFS Phase V values.

In [ ]:
from ngfs_data import load_ngfs_scenarios

result = load_ngfs_scenarios(use_api=False, seed=42)
scenarios = result["scenarios"]
print(f"Source: {result['source']}")
print(f"Pathways: {list(scenarios.keys())}")

# Display one scenario
scenarios["net_zero_2050"].head()

In [ ]:
# Carbon price trajectories
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, df in scenarios.items():
    axes[0].plot(df["year"], df["carbon_price"], label=name, linewidth=1.5)
    axes[1].plot(df["year"], df["temperature_anomaly"], label=name, linewidth=1.5)

axes[0].set_title("Carbon Price Trajectories (USD/tCO2)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Carbon Price ($/tCO2)")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Temperature Anomaly (C above pre-industrial)")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Temperature Anomaly (C)")
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 2. Sector Carbon Intensity Overview

Carbon intensity measures emissions per unit of revenue. High-intensity sectors (energy, utilities, materials) face the largest transition risk.

In [ ]:
from ngfs_data import get_sector_carbon_intensity

sector_data = get_sector_carbon_intensity(seed=42)
display(sector_data)

fig, ax = plt.subplots(figsize=(10, 5))
sector_data["carbon_intensity"].sort_values(ascending=True).plot.barh(ax=ax, color="#4472C4")
ax.set_xlabel("Carbon Intensity (tCO2/$M revenue)")
ax.set_title("Sector Carbon Intensity")
fig.tight_layout()
plt.show()

## 3. Transition Risk: Sector Repricing

Under each scenario, sectors face carbon costs that reduce equity value. The equity impact is computed as the ratio of carbon cost to EBITDA.

In [ ]:
from transition_risk import sector_repricing, transition_loss_by_scenario

weights = np.array([0.10, 0.10, 0.10, 0.15, 0.15, 0.20, 0.10, 0.10])

# Example: Net Zero 2050 at carbon price $250
repriced = sector_repricing(sector_data, carbon_price=250.0)
print("Sector repricing under Net Zero 2050 ($250/tCO2):")
display(repriced[["carbon_intensity", "carbon_cost", "equity_impact"]])

# All scenarios
trans_df = transition_loss_by_scenario(sector_data, scenarios, weights)
print("\nTransition losses by scenario:")
display(trans_df)

## 4. Physical Risk: Damage Functions and Sector Exposure

Physical risk is modeled using the Nordhaus quadratic damage function. Sector-level multipliers reflect differential exposure to climate hazards.

In [ ]:
from physical_risk import temperature_damage_function, physical_risk_by_sector, physical_loss_by_scenario

# Damage function curve
temps = np.linspace(0, 5, 100)
damages = [temperature_damage_function(t) for t in temps]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(temps, damages, linewidth=2, color="#C44E52")
axes[0].set_xlabel("Temperature Anomaly (C)")
axes[0].set_ylabel("GDP Fraction Lost")
axes[0].set_title("Nordhaus Quadratic Damage Function")
axes[0].grid(True, alpha=0.3)

# Sector exposure at 3C
risks = physical_risk_by_sector(3.0)
risk_s = pd.Series(risks).sort_values(ascending=True)
risk_s.plot.barh(ax=axes[1], color="#C44E52")
axes[1].set_xlabel("Physical Risk Damage (fraction)")
axes[1].set_title("Sector Physical Risk at 3C Warming")

fig.tight_layout()
plt.show()

# Physical losses by scenario
phys_df = physical_loss_by_scenario(sector_data, scenarios, weights)
print("Physical losses by scenario:")
display(phys_df)

## 5. Climate VaR: Combined Risk

Climate VaR combines transition and physical risks across all scenarios, taking the 95th percentile of total losses.

In [ ]:
from model import ClimateRiskModel

model = ClimateRiskModel()
model.load_data(use_api=False)

climate_var = model.compute_climate_var(alpha=0.95)
print(f"Climate VaR (95%): {climate_var['var']:.4f}")
print(f"Climate ES (95%): {climate_var['es']:.4f}")

# Scenario comparison
comparison = model.scenario_comparison()
display(comparison)

# Visualize
from diagnostics import plot_scenario_heatmap
fig, ax = plot_scenario_heatmap(comparison)
plt.show()

## 6. Sobol Sensitivity Analysis

Global sensitivity analysis identifies which climate factors (carbon price, temperature, GDP impact, sea-level rise) drive the most variance in portfolio losses.

In [ ]:
sobol_df = model.run_sobol()
display(sobol_df)

from diagnostics import plot_sobol_tornado
fig, ax = plot_sobol_tornado(sobol_df)
plt.show()

## 7. TCFD Metrics and Report

Generate a TCFD-aligned markdown report with key metrics (WACI, financed emissions) and scenario comparison tables.

In [ ]:
from tcfd_metrics import compute_waci_path, compute_financed_emissions
from transition_risk import waci

# Current WACI
current_waci = waci(weights, sector_data["carbon_intensity"].values)
print(f"Current WACI: {current_waci:.1f} tCO2/$M revenue")

# Financed emissions for $1B portfolio
fe = compute_financed_emissions(weights, sector_data["carbon_intensity"].values, portfolio_value=1000.0)
print(f"Financed Emissions: {fe:,.0f} tCO2")

# WACI evolution across scenarios
waci_frames = []
for name, sdf in scenarios.items():
    waci_path = compute_waci_path(weights, sector_data["carbon_intensity"].values, sdf)
    waci_path["scenario"] = name
    waci_frames.append(waci_path)

waci_all = pd.concat(waci_frames, ignore_index=True)

from diagnostics import plot_waci_evolution
fig, ax = plot_waci_evolution(waci_all)
plt.show()

In [ ]:
# Full TCFD report
from IPython.display import Markdown

report = model.tcfd_summary()
display(Markdown(report))

In [ ]:
# Stranded assets visualization
from diagnostics import plot_stranded_assets

fig, ax = plot_stranded_assets(sector_data, carbon_prices=[50, 100, 200, 400])
plt.show()